# Fine-Tuning Whisper Small for Arabic ASR

This notebook provides a complete, end-to-end pipeline for fine-tuning the `openai/whisper-small` model for Automatic Speech Recognition (ASR) using the Arabic Speech Corpus. Optimized for Google Colab, it walks through dataset extraction, Arabic text normalization, memory-efficient audio processing, and robust Seq2Seq training. The workflow is designed to prevent common gradient checkpointing errors and automatically saves the best-performing checkpoints directly to Google Drive.

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Download Arabic Speech Corpus Dataset

In [ ]:
import requests, os, zipfile
from tqdm import tqdm

url = "http://www.arabicspeechcorpus.com/arabic-speech-corpus.zip"
zip_path = "arabic-speech-corpus.zip"
extract_path = "arabic-speech-corpus"

def download_dataset(url, save_path):
    existing_size = os.path.getsize(save_path) if os.path.exists(save_path) else 0
    headers = {"Range": f"bytes={existing_size}-"}
    response = requests.get(url, headers=headers, stream=True, timeout=60)
    mode = "ab" if existing_size > 0 else "wb"
    total_size = int(response.headers.get('content-length', 0)) + existing_size

    with open(save_path, mode) as f, tqdm(total=total_size, initial=existing_size, unit='B', unit_scale=True) as bar:
        for chunk in response.iter_content(chunk_size=1024*1024):
            if chunk:
                f.write(chunk)
                bar.update(len(chunk))

if not os.path.exists(extract_path):
    download_dataset(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    os.remove(zip_path)
    print("Done!")
else:
    print("Dataset already extracted.")

## Step 3: Locate Dataset Transcript Path

In [ ]:
import os

def find_transcript(root_dir):
    for root, dirs, files in os.walk(root_dir):
        if "orthographic-transcript.txt" in files:
            return root
    return None

correct_path = find_transcript(".")

if correct_path:
    print(f"Dataset located at: {correct_path}")
    corpus_folder = correct_path   # ← THIS is what defines corpus_folder
else:
    print("Transcript not found.")
    print("Files in current directory:", os.listdir("."))

## Step 4: Install Dependencies (Restart Required)

In [ ]:
!pip uninstall transformers peft accelerate -y -q
!pip install transformers==4.44.2 accelerate==0.33.0 datasets evaluate jiwer librosa -q

import os
os._exit(0)  # Force restart kernel

## Step 5: Load Whisper Model & Processor

In [ ]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor, WhisperForConditionalGeneration

model_id = "openai/whisper-small"

# 1. Load the helper tools
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_id)
tokenizer = WhisperTokenizer.from_pretrained(model_id, language="arabic", task="transcribe")
processor = WhisperProcessor.from_pretrained(model_id, language="arabic", task="transcribe")

# 2. Load the model from scratch to completely clear the memory
model = WhisperForConditionalGeneration.from_pretrained(model_id)

# 🌟 Magic step: Force the model to completely unfreeze the weights (Unfreeze Everything)
for param in model.parameters():
    param.requires_grad = True

# 🌟 Solve the mandatory warning issue to enable gradients with checkpointing
model.enable_input_require_grads()

# 3. Adjust generation settings in generation_config
model.generation_config.language = "arabic"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="arabic", task="transcribe")
model.generation_config.suppress_tokens = []

# Clear values from the old config to ensure no conflicts (Error Fix)
model.config.suppress_tokens = None
model.config.forced_decoder_ids = None
model.config.use_cache = False

print("✅ Model reloaded clean and completely UNFROZEN!")

## Step 6: Arabic Text Normalization Utilities

In [ ]:
import re

def buckwalter_to_arabic(text):
    bw_map = {
        "'": "ء", "|": "آ", ">": "أ", "&": "ؤ", "<": "إ", "}": "ئ",
        "A": "ا", "b": "ب", "p": "ة", "t": "ت", "v": "ث", "j": "ج",
        "H": "ح", "x": "خ", "d": "د", "*": "ذ", "r": "ر", "z": "ز",
        "s": "س", "$": "ش", "S": "ص", "D": "ض", "T": "ط", "Z": "ظ",
        "E": "ع", "g": "غ", "f": "ف", "q": "ق", "k": "ك", "l": "ل",
        "m": "م", "n": "ن", "h": "ه", "w": "و", "y": "ي",
        "F": "ً", "N": "ٌ", "K": "ٍ", "a": "َ", "u": "ُ", "i": "ِ",
        "~": "ّ", "o": "ْ", "-": "", "_": ""
    }
    return "".join([bw_map.get(c, c) for c in text])

def normalize_arabic(text):
    # Convert from Buckwalter to Arabic first
    text = buckwalter_to_arabic(text)

    # Remove diacritics  (التشكيل)
    diacritics = re.compile(r'[\u064B-\u0652]')
    text = re.sub(diacritics, '', text)

    # Normalize ا, ي, and ة
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ة", "ه", text)

    # Keep only Arabic letters and spaces
    text = re.sub(r'[^\u0621-\u064A\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

## Step 7: Load Data List (First Attempt)

In [ ]:
import os

# Use the path where cell number 7 found the data
wav_dir = os.path.join(corpus_folder, "wav")
transcript_file = os.path.join(corpus_folder, "orthographic-transcript.txt")

data_list = []
if os.path.exists(transcript_file):
    with open(transcript_file, 'r', encoding='utf-8') as f:
        for line in f:
            # Split the line to get the filename and text (the first file, then the text between quotes)
            parts = line.strip().split(' "')
            if len(parts) >= 2:
                wav_filename = parts[0].replace('"', '').split('/')[-1]
                if not wav_filename.endswith('.wav'):
                    wav_filename += '.wav'
                full_wav_path = os.path.join(wav_dir, wav_filename)

                # Add the sample only if the audio file actually exists
                if os.path.exists(full_wav_path):
                    data_list.append({
                        "audio": full_wav_path,
                        "sentence": normalize_arabic(parts[1].replace('"', ''))
                    })

print(f"Total valid samples found: {len(data_list)}")

## Step 8: Debug — Verify Dataset Structure

In [ ]:
import os

print("=== corpus_folder ===")
print(corpus_folder)

print("\n=== Files inside corpus_folder ===")
print(os.listdir(corpus_folder))

print("\n=== Checking wav folder ===")
wav_path = os.path.join(corpus_folder, "wav")
if os.path.exists(wav_path):
    wav_files = os.listdir(wav_path)
    print(f"✅ wav folder exists — {len(wav_files)} files")
    print("First 3 files:", wav_files[:3])
else:
    print("❌ wav folder NOT found!")

print("\n=== Checking transcript file ===")
transcript_path = os.path.join(corpus_folder, "orthographic-transcript.txt")
if os.path.exists(transcript_path):
    print("✅ Transcript file exists")
    with open(transcript_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    print(f"Total lines: {len(lines)}")
    print("First 2 lines:")
    for line in lines[:2]:
        print(repr(line))
else:
    print("❌ Transcript file NOT found!")

## Step 9: Load Data List (Corrected Parser)

In [ ]:
import os

wav_dir = os.path.join(corpus_folder, "wav")
transcript_file = os.path.join(corpus_folder, "orthographic-transcript.txt")

data_list = []

with open(transcript_file, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        # Format: "filename.wav" "transcription text"
        # Split into exactly 2 parts by the quote pattern
        parts = line.split('" "')
        if len(parts) != 2:
            continue

        wav_filename = parts[0].replace('"', '').strip()
        transcription = parts[1].replace('"', '').strip()

        # Make sure it ends with .wav
        if not wav_filename.endswith('.wav'):
            wav_filename += '.wav'

        full_wav_path = os.path.join(wav_dir, wav_filename)

        if os.path.exists(full_wav_path):
            data_list.append({
                "audio": full_wav_path,
                "sentence": normalize_arabic(transcription)
            })
        else:
            print(f"⚠️ Not found: {full_wav_path}")

print(f"✅ Total valid samples found: {len(data_list)}")
if len(data_list) > 0:
    print(f"Example: {data_list[0]}")

## Step 10: Prepare HuggingFace Dataset & Feature Extraction

In [ ]:
from datasets import Dataset, Audio

print("🚀 Loading raw data...")
raw_dataset = Dataset.from_list(data_list)

# 🌟 Magic solution: We'll let the library read the audio in its highly optimized way and resample to 16000
raw_dataset = raw_dataset.cast_column("audio", Audio(sampling_rate=16000))

def prepare_dataset(batch):
    # The audio is now read automatically by the library and we get the ready array
    audio = batch["audio"]

    # Extract Input Features
    batch["input_features"] = processor.feature_extractor(audio["array"], sampling_rate=16000).input_features[0]

    # Extract text (Labels)
    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids

    return batch

print("⏳ Running highly safe processing (without Multiprocessing or librosa)...")
# We entirely disabled num_proc to prevent crashes
processed_dataset = raw_dataset.map(
    prepare_dataset,
    remove_columns=["audio", "sentence"]
)

print("✂️ Splitting the data...")
dataset_split = processed_dataset.train_test_split(test_size=0.1)

print(f"\n✅ Processing completed successfully without any crashes!")
print(f"🔥 Training samples: {len(dataset_split['train'])}")
print(f"🧪 Evaluation samples: {len(dataset_split['test'])}")

## Step 11: Define Custom Data Collator

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # 1. Process audio features
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # 2. Prepare labels and clean them from the repeated first token (50258)
        cleaned_labels = []
        for feature in features:
            label = feature["labels"]
            # If the label starts with 50258 (startoftranscript), remove it to prevent repetition
            if len(label) > 0 and label[0] == 50258:
                cleaned_labels.append(torch.tensor(label[1:]))
            else:
                cleaned_labels.append(torch.tensor(label))

        # Pad the labels with -100
        labels = torch.nn.utils.rnn.pad_sequence(cleaned_labels, batch_first=True, padding_value=-100)
        batch["labels"] = labels

        # 3. 🔥 Radical solution for ValueError: manually and correctly generating decoder_input_ids here
        # This completely eliminates the gradient checkpointing issue
        decoder_input_ids = labels.clone()
        # Temporarily replace -100 with pad_token_id to pass it to the decoder
        decoder_input_ids[decoder_input_ids == -100] = self.processor.tokenizer.pad_token_id

        # Safe manual Shift Right
        decoder_input_ids[:, 1:] = decoder_input_ids[:, :-1].clone()
        decoder_input_ids[:, 0] = 50258  # Place the start token <|startoftranscript|> in the first column

        batch["decoder_input_ids"] = decoder_input_ids

        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print("🎯 Data Collator is fully optimized with manual decoder alignment!")

## Step 12: Verify Data Collator Output

In [ ]:
# Inspect the first training sample
sample_batch = data_collator([dataset_split["train"][0]])
print(f"Input Features Shape: {sample_batch['input_features'].shape}") # Should be (1, 80, 3000)
print(f"Labels (first 10 tokens): {sample_batch['labels'][0][:10]}") # Should contain numbers, not just -100

## Step 13: Remove Conflicting Packages

In [ ]:
# Uninstall the library causing the conflict
!pip uninstall peft -y -q

# Minor update to transformers to ensure all new classes are available
!pip install transformers==4.44.2 --quiet

## Step 14: Define Training Arguments (Initial Pass)

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/whisper-arabic-checkpoints", # 🌟 Modification here: Save directly to Drive
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=5e-6,
    max_steps=400,      # More than enough for this data size
    warmup_steps=50,
    weight_decay=0.01,
    label_smoothing_factor=0.1, # Strongly prevents Overfitting
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="epoch",        # Changed to evaluate after each epoch
    save_strategy="epoch",        # Changed to save (Checkpoint) after each epoch
    save_total_limit=2,           # Deletes old ones and keeps only the last two versions to save disk space
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    generation_num_beams=5,
    logging_steps=50,
    report_to=["tensorboard"],
    load_best_model_at_end=True,  # Will load the best version at the end based on WER
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
)

print("Training arguments are set")

## Step 15: Fix Generation Configuration

In [ ]:
# Move generation parameters to the correct config object
model.generation_config.suppress_tokens = []
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="arabic", task="transcribe")

# Clear the old parameters from model.config to stop the ValueError
model.config.suppress_tokens = None
model.config.forced_decoder_ids = None

print("Generation configuration fixed")

## Step 16: Freeze Encoder & Enable SpecAugment

In [ ]:
# 1. Freeze the Encoder completely
model.freeze_encoder()

# 2. Activate SpecAugment from the Config
model.config.apply_spec_augment = True
model.config.mask_time_prob = 0.05
model.config.mask_feature_prob = 0.05

print("Encoder frozen and SpecAugment activated!")

## Step 17: Build Trainer & Run Training

In [ ]:
import evaluate
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback

metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    # Clean the outputs before calculation to ensure fairness
    pred_str = [normalize_arabic(p) for p in pred_str]
    label_str = [normalize_arabic(l) for l in label_str]

    final_preds, final_labels = [], []
    for p, l in zip(pred_str, label_str):
        if len(l.strip()) > 0:
            final_preds.append(p)
            final_labels.append(l)

    if not final_labels:
        return {"wer": 100.0}

    wer = 100 * metric.compute(predictions=final_preds, references=final_labels)
    return {"wer": wer}

# Stable and tested training arguments for Unfrozen Whisper
training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/whisper-arabic-checkpoints", # 🌟 Modification here: Save directly to Drive
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    max_steps=600,
    warmup_steps=50,
    weight_decay=0.01,
    label_smoothing_factor=0.1,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    generation_num_beams=5,       # To activate Beam Search and prevent repetition and hallucination
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
)

# Resolve conflicts in the Config
model.generation_config.language = "arabic"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []

model.config.suppress_tokens = None
model.config.forced_decoder_ids = None
model.config.use_cache = False

# Build the clean Trainer
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset_split["train"],
    eval_dataset=dataset_split["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("🚀 Starting REAL Training with Perfect Manual Alignment...")
trainer.train()

## Step 18: Save Final Model to Google Drive

In [ ]:
print("🚀 Starting REAL Training...")
trainer.train()

# 🌟 Final save to Google Drive so it never gets lost
final_model_path = "/content/drive/MyDrive/whisper-arabic-final-sota"
trainer.save_model(final_model_path)
processor.save_pretrained(final_model_path)

print(f"✅ Training Finished! Golden Model saved cleanly to your Google Drive: {final_model_path}")

In [1]:
!pip freeze > requirements.txt